# 🧹 آموزش تمیز کردن داده

## روش‌های تشخیص اوتلایر (Outlier Detection)

### 📊 روش IQR (Interquartile Range)

در این روش از فاصله بین چارک اول (Q1) و چارک سوم (Q3) استفاده می‌کنیم.

**فرمول:**
- IQR = Q3 - Q1
- حد پایین = Q1 - 1.5 × IQR
- حد بالا = Q3 + 1.5 × IQR

هر داده‌ای خارج از این محدوده، اوتلایر محسوب می‌شود.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ساختن دیتای نمونه
np.random.seed(42)
data = np.random.normal(100, 15, 200)
outliers = np.array([200, 210, 15, 8, 250, 5])
data_with_outliers = np.concatenate([data, outliers])

df = pd.DataFrame({'value': data_with_outliers})
print(f"تعداد کل داده‌ها: {len(df)}")
df.head(10)

In [ ]:
# تشخيص اوتلایر با روش IQR
Q1 = df['value'].quantile(0.25)
Q3 = df['value'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_iqr = df[(df['value'] < lower_bound) | (df['value'] > upper_bound)]

print(f"Q1: {Q1:.2f}")
print(f"Q3: {Q3:.2f}")
print(f"IQR: {IQR:.2f}")
print(f"حد پایین: {lower_bound:.2f}")
print(f"حد بالا: {upper_bound:.2f}")
print(f"تعداد اوتلایرها: {len(outliers_iqr)}")
print(f"\nاوتلایرهای پیدا شده:")
print(outliers_iqr)

In [ ]:
# حذف اوتلایرها
df_cleaned_iqr = df[(df['value'] >= lower_bound) & (df['value'] <= upper_bound)]

print(f"قبل از حذف: {len(df)} رکورد")
print(f"بعد از حذف: {len(df_cleaned_iqr)} رکورد")
print(f"حذف شد: {len(df) - len(df_cleaned_iqr)} رکورد")

### 📈 روش Z-Score (Standard Score)

Z-Score فاصله هر داده را از میانگین بر حسب انحراف معیار نشان می‌دهد.

**فرمول:** Z = (x - μ) / σ

داده‌هایی با |Z| > 3 معمولاً اوتلایر در نظر گرفته می‌شوند.

In [ ]:
# محاسبه Z-Score
z_scores = np.abs(stats.zscore(df['value']))
outliers_zscore = df[z_scores > 3]

print(f"تعداد اوتلایرها با Z-Score > 3: {len(outliers_zscore)}")
print(f"\nاوتلایرهای پیدا شده:")
print(outliers_zscore)

In [ ]:
# حذف اوتلایرها با Z-Score
df_cleaned_zscore = df[z_scores <= 3]

print(f"قبل از حذف: {len(df)} رکورد")
print(f"بعد از حذف: {len(df_cleaned_zscore)} رکورد")
print(f"حذف شد: {len(df) - len(df_cleaned_zscore)} رکورد")

In [ ]:
# مقایسه بصری قبل و بعد از حذف اوتلایرها
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(y=df['value'], ax=axes[0], color='lightcoral')
axes[0].set_title('قبل از حذف اوتلایر')

sns.boxplot(y=df_cleaned_iqr['value'], ax=axes[1], color='lightgreen')
axes[1].set_title('بعد از حذف اوتلایر (روش IQR)')

plt.tight_layout()
plt.show()

## 🎯 خلاصه

| روش | مزایا | معایب | بهترین کاربرد |
|-----|-------|-------|---------------|
| IQR | به توزیع داده حساس نیست، برای داده‌های غیرنرمال خوب است | آستانه 1.5 ممکن است همیشه مناسب نباشد | داده‌های با توزیع نامشخص یا چوله (skewed) |
| Z-Score | دقیق و استاندارد است | به نرمال بودن توزیع داده حساس است | داده‌های با توزیع نرمال یا تقریباً نرمال |